# Word Embedding

This notebook reproduces the second step of pipeline described in: *"Enhancing comorbidity network inference with risk-enriched health trajectories embedding"*

> **Pre-trained model**
>
> This section requires the BioWordVec base model (`BioWordVec/pubmed_mesh_test`).
> If you have not already trained it, all the files neede are available in the BioWordVec folder.
> Run the following from terminal **before** executing this section:
>
> ```bash
> cd BioWordVec
> python BioWordVec.py
> ```
>
> Training takes ~15–30 minutes depending on hardware. The model files
> (`pubmed_mesh_test`, `pubmed_mesh_test.bin`) will be saved in the BioWordVec
> directory.

## Section 0 — Data Import

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
from scipy.spatial.distance import cosine
from scipy.stats import mannwhitneyu, pearsonr, spearmanr
from scipy.special import erfc
from sklearn.covariance import GraphicalLassoCV
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120})

# Output directories
OUTPUT_RES_DIR   = "results"
OUTPUT_PLOTS_DIR = "plots"
os.makedirs(OUTPUT_RES_DIR,   exist_ok=True)
os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)

In [2]:
sequence_main_final = pd.read_csv(os.path.join(OUTPUT_RES_DIR,'sequence_main_final.csv'))
sequence_par_final = pd.read_csv(os.path.join(OUTPUT_RES_DIR,'sequence_par_final.csv'))
sequence_main_par_final =  pd.read_csv(os.path.join(OUTPUT_RES_DIR,'sequence_main_par_final.csv'))

In [3]:
sequence_main_final.head()

,ID,age_in_days_since_born,event_name
0,1,20976,Diabetes Mellitus
1,1,21997,Hypertension
2,1,28480,"Kidney Diseases, Chronic, Stage 3"
3,1,29402,"Kidney Diseases, Chronic, Stage 3"
4,2,17678,Diabetes Mellitus


In [4]:
DISEASE_NAMES_MAIN = [n for n in sequence_main_final["event_name"].unique()]
RF_NAMES=[n for n in sequence_par_final["event_name"].unique()]
EVENTS_NAMES = DISEASE_NAMES_MAIN + RF_NAMES   # full vocabulary
print(f"Total vocabulary size: {len(EVENTS_NAMES)}")

Total vocabulary size: 103


In [5]:
from pathlib import Path
from gensim.models import FastText
import time
PRETRAINED_MODEL_PATH  = "./BioWordVec/pubmed_mesh_test"
FINETUNED_MAIN_PATH    = os.path.join(OUTPUT_RES_DIR,"pubmed_mesh_finetuned_main")
FINETUNED_MAINPAR_PATH = os.path.join(OUTPUT_RES_DIR,"pubmed_mesh_finetuned_main_par")

if not Path(PRETRAINED_MODEL_PATH).exists():
    raise FileNotFoundError(
        "Pre-trained model not found at './BioWordVec/pubmed_mesh_test'.\n"
        "Run:\n"
        "cd BioWordVec\n"
        "python BioWordVec.py\n"
        "then re-execute this cell."
    )
print("Pre-trained BioWordVec model found.")
base_model = FastText.load(PRETRAINED_MODEL_PATH)
print(f"  Vector size: {base_model.vector_size}  |  Vocab size: {len(base_model.wv)}")


Pre-trained BioWordVec model found.
  Vector size: 200  |  Vocab size: 65158


## Section 1 — Fine-tuning of BioWordVec

In [6]:
def build_corpus(seq_df: pd.DataFrame, phrase_col: str = "event_name") -> list:
    corpus = []
    for pid, grp in seq_df.sort_values("age_in_days_since_born").groupby("ID"):
        tokens = []
        for phrase in grp[phrase_col]:
            # Split multi-word MeSH terms into individual tokens
            tokens.extend(phrase.replace(",", " ").lower().split())
        if tokens:
            corpus.append(tokens)
    return corpus

corpus_main    = build_corpus(sequence_main_final)
corpus_mainpar = build_corpus(sequence_main_par_final)
print(f"Corpus (main):         {len(corpus_main):,} sentences")
print(f"Corpus (main+parallel): {len(corpus_mainpar):,} sentences")
print(f"Example sentence (main): {corpus_main[0][:10]} …")

Corpus (main):         4,469 sentences
Corpus (main+parallel): 4,469 sentences
Example sentence (main): ['diabetes', 'mellitus', 'hypertension', 'kidney', 'diseases', 'chronic', 'stage', '3', 'kidney', 'diseases'] …


### 1.1  Fine-tune on main sequence

> **Fine-tuning hyperparameters**
>
> Adjust the `FINETUNE_EPOCHS` parameter below before running the fine-tuning cells.
>
> Note: More epochs → better sequence adaptation, less domain contribution, longer runtime

In [7]:
# In the paper choosed based on embedding-similarity between original and the model fine-tuned on main sequences
FINETUNE_EPOCHS  = 4    

In [8]:
# Check vocabulary coverage
all_words = [word for seq in corpus_main for word in seq]
unique_words = sorted(set(all_words))
base_model_words = list(base_model.wv.key_to_index)
missing_words = [w for w in unique_words if w not in base_model_words]

print(f"Number of unique corpus words: {len(unique_words)}")
print(f"Number of missing words in pretrained model: {len(missing_words)}")

if missing_words:
    print("\nMissing words:")
    print(missing_words)

Number of unique corpus words: 39
Number of missing words in pretrained model: 1

Missing words:
['endstage']


In [9]:
import copy
print("Fine-tuning on main sequence …")
model_main = copy.deepcopy(base_model)
model_main.build_vocab(corpus_main, update=True)
for epoch in range(FINETUNE_EPOCHS):
    model_main.train(corpus_main,
                     total_examples=len(corpus_main),
                     epochs=1)
    print(f"Epoch {epoch + 1} completed.")
model_main.save(FINETUNED_MAIN_PATH)
print(f"  Saved → {FINETUNED_MAIN_PATH}")

Fine-tuning on main sequence …
Epoch 1 completed.
Epoch 2 completed.
Epoch 3 completed.
Epoch 4 completed.
  Saved → results/pubmed_mesh_finetuned_main


In [11]:
# Define the epoch to fine-tuned on based on the mean cosing beeing over 75% for the words present in our sequences
target_words = list(set(all_words))

comparison = []

for word in target_words:
    if word in base_model.wv and word in model_main.wv:
        original_vec = base_model.wv[word]
        updated_vec = model_main.wv[word]
        similarity = 1 - cosine(original_vec, updated_vec)
        comparison.append((word, similarity))
    elif word in updated_model.wv:
        comparison.append((word, "Added during fine-tuning)"))
    else:
        comparison.append((word, "Not in fine-tuned model"))

df_compare = pd.DataFrame(comparison, columns=["Word", "Cosine Similarity"])
numeric_similarities = pd.to_numeric(df_compare["Cosine Similarity"], errors='coerce')
mean_similarity = numeric_similarities.mean()
print(f"Mean cosine similarity (original vs fine-tuned): {mean_similarity:.4f}")
#### Epochs defined so that the cosine similarity of the word embedding between original and fine-tuned is over 70%

Mean cosine similarity (original vs fine-tuned): 0.8987


### 1.2  Fine-tune on main and parallel sequence

In [12]:
# In the paper we used the same one as the main sequence
FINETUNE_EPOCHS  = 4    

In [13]:
# Check vocabulary coverage
all_words = [word for seq in corpus_mainpar for word in seq]
unique_words = sorted(set(all_words))
base_model_words = list(base_model.wv.key_to_index)
missing_words = [w for w in unique_words if w not in base_model_words]

print(f"Number of unique corpus words: {len(unique_words)}")
print(f"Number of missing words in pretrained model: {len(missing_words)}")

if missing_words:
    print("\nMissing words:")
    print(missing_words)

Number of unique corpus words: 121
Number of missing words in pretrained model: 7

Missing words:
['bnp', 'endstage', 'hba1c', 'mid-range', 'nt-probnp', 'widowed', 'yes']


In [14]:
print("Fine-tuning on main sequence …")
model_mainpar = copy.deepcopy(base_model)
model_mainpar.build_vocab(corpus_mainpar, update=True)
for epoch in range(FINETUNE_EPOCHS):
    model_mainpar.train(corpus_mainpar,
                     total_examples=len(corpus_mainpar),
                     epochs=1)
    print(f"Epoch {epoch + 1} completed.")
model_mainpar.save(FINETUNED_MAINPAR_PATH)
print(f"  Saved → {FINETUNED_MAINPAR_PATH}")

Fine-tuning on main sequence …
Epoch 1 completed.
Epoch 2 completed.
Epoch 3 completed.
Epoch 4 completed.
  Saved → results/pubmed_mesh_finetuned_main_par


## Section 2 — Extract events embeddings

For each disease / risk-factor phrase, we average its word-level embeddings to obtain a
phrase-level vector.

In [15]:
def get_events_embeddings(phrases: list, model) -> dict:
    phrase_embeddings = {}
    for phrase in phrases:
        tokens = phrase.replace(",", " ").replace("-", " ").lower().split()
        vecs = []
        for token in tokens:
            if token in model.wv:
                vecs.append(model.wv[token])
            else:
                print(f"Word '{token}' not in vocabulary, skipped.")
        if vecs:
            phrase_embeddings[phrase] = np.mean(vecs, axis=0)
        else:
            print(f"No valid embeddings found for phrase: '{phrase}'")
    return phrase_embeddings

In [16]:
phrase_embeddings_main = get_events_embeddings(DISEASE_NAMES_MAIN, model_main)
phrase_emb_main_df = pd.DataFrame.from_dict(
    phrase_embeddings_main,
    orient="columns"
)
phrase_emb_main_df.to_csv(os.path.join(OUTPUT_RES_DIR,"cardiomet_embeddings.csv"), index=False)
print(f"  Saved → cardiomet_embeddings.csv")

  Saved → cardiomet_embeddings.csv


In [17]:
phrase_embeddings_mainpar = get_events_embeddings(EVENTS_NAMES, model_mainpar)
phrase_emb_mainpar_df = pd.DataFrame.from_dict(
    phrase_embeddings_mainpar,
    orient="columns"
)
phrase_emb_mainpar_df.to_csv(os.path.join(OUTPUT_RES_DIR,"cardiomet_risk_adj_embeddings.csv"), index=False)
print(f"  Saved → cardiomet_risk_adj_embeddings.csv")

  Saved → cardiomet_risk_adj_embeddings.csv
